# Downstream of CaMKII

## LCC phosphorylated by CaMKII

In the Morrotti model:

```julia
k_ckLCC = 0.4                   # [s^-1]
k_pp1LCC = 0.1103               # [s^-1]
k_pp2aLCC = 10.1                # [s^-1]
KmCK_LCC = 12                   # [uM]
KmPKA_LCC = 21                  # [uM]
KmPP2A_LCC = 47                 # [uM]
KmPP1_LCC = 9                   # [uM]
LCCtotSL = 0.0846          # [uM] - Total Subsarcolemmal [LCC] (umol/l sl)
PP1_SL = 0.57              # [uM] - Total Subsarcolemmal [PP1]
CaMKIItotSL = 120 * 8.293e-4      # [uM]
CaMKIIact_SL = CaMKIItotSL .* (Pb_sl + Pt_sl + Pt2_sl + Pa_sl)
D(LCC_CKslp) ~ (LCCSL_PHOS - LCCSL_DEPHOS)
LCC_CKsln = LCCtotSL - LCC_CKslp
LCCSL_PHOS = (k_ckLCC * CaMKIIact_SL * LCC_CKsln) / (KmCK_LCC + LCC_CKsln)
LCCSL_DEPHOS = (k_pp1LCC * PP1_SL * LCC_CKslp) / (KmPP1_LCC + LCC_CKslp)
```

In [ ]:
using ModelingToolkit
using ModelingToolkit: t_nounits as t, D_nounits as D
using OrdinaryDiffEq
using SteadyStateDiffEq
using Plots

In [ ]:
function lcc_ckp(;name)
    @parameters begin
        CaMKIIact_SL = 1
        k_ckLCC = 0.4                   # [s^-1]
        k_pp1LCC = 0.1103               # [s^-1]
        k_pp2aLCC = 10.1                # [s^-1]
        KmCK_LCC = 12                   # [uM]
        KmPKA_LCC = 21                  # [uM]
        KmPP2A_LCC = 47                 # [uM]
        KmPP1_LCC = 9                   # [uM]
        LCCtotSL = 0.0846          # [uM] - Total Subsarcolemmal [LCC] (umol/l sl)
        PP1_SL = 0.57              # [uM] - Total Subsarcolemmal [PP1]
        CaMKIItotSL = 120 * 8.293e-4      # [uM]
    end
    @variables begin
        LCC_CKslp(t) = 0
        LCC_CKsln(t)
        fracLCCCkp(t)
    end

    LCCSL_PHOS = (k_ckLCC * CaMKIIact_SL * LCC_CKsln) / (KmCK_LCC + LCC_CKsln)
    LCCSL_DEPHOS = (k_pp1LCC * PP1_SL * LCC_CKslp) / (KmPP1_LCC + LCC_CKslp)

    eqs = [
        LCCtotSL ~ LCC_CKslp + LCC_CKsln,
        D(LCC_CKslp) ~ LCCSL_PHOS - LCCSL_DEPHOS,
        fracLCCCkp ~ LCC_CKslp / LCCtotSL
    ]

    return System(eqs, t; name)
end

In [ ]:
@mtkbuild sys = lcc_ckp()

In [ ]:
ssprob = SteadyStateProblem(sys, [])
sol = solve(ssprob, DynamicSS(KenCarp4()))

In [ ]:
sol[sys.fracLCCCkp]